# determl GPU Verification Test

This notebook tests determl on a real GPU to prove it enforces determinism where it actually matters.

**CPU is already deterministic by nature.** The real test is GPU — where Flash Attention, CUDA thread ordering, and cuDNN algorithm selection introduce non-determinism.

---

## Step 1: Check GPU

In [ ]:
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name: {torch.cuda.get_device_name(0)}')
    print(f'CUDA version: {torch.version.cuda}')
    print(f'PyTorch version: {torch.__version__}')
else:
    print('NO GPU! Go to Runtime > Change runtime type > T4 GPU')

## Step 2: Install determl

In [ ]:
!pip install git+https://github.com/xailong-6969/determl.git@v2-enforcement -q
!pip install transformers accelerate -q
print('determl installed!')

## Step 3: Test WITHOUT determl (prove non-determinism exists)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'Qwen/Qwen2.5-Coder-0.5B-Instruct'
print(f'Loading {model_name}...')

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
model.to('cuda')
model.eval()

prompt = 'What is 2 + 2? Answer with just the number.'

# Run 5 times WITHOUT any determinism controls
print('\n--- WITHOUT determl ---')
hashes = []
for i in range(5):
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=50, do_sample=False)
    output_hash = hash(output.cpu().numpy().tobytes())
    hashes.append(output_hash)
    print(f'  Run {i+1}: hash={output_hash}')

if len(set(hashes)) == 1:
    print('\nResult: All same (greedy on this GPU happens to be deterministic)')
    print('But the INTERNAL tensor values may still differ slightly!')
else:
    print(f'\nResult: NON-DETERMINISTIC! {len(set(hashes))} different hashes')

# Clean up
del model, tokenizer
torch.cuda.empty_cache()

## Step 4: Test WITH determl (prove determinism is enforced)

In [ ]:
from determl import DeterministicEngine

# Load with full enforcement
engine = DeterministicEngine(seed=42, precision='high')
report = engine.load(model_name)

print('--- Enforcement Report ---')
print(report)
print(f'\nDevice: {engine.device}')
print(f'Engine: {engine}')

In [ ]:
# Verify determinism — 5 runs
print('\n--- WITH determl ---')
result = engine.verify(prompt='What is 2 + 2? Answer with just the number.', num_runs=5)
print(result)

## Step 5: Test with MULTIPLE prompts

In [ ]:
test_prompts = [
    'What is 2 + 2? Answer with just the number.',
    'Write hello world in Python.',
    'What is the capital of France? One word.',
    'Explain gravity in one sentence.',
    'Translate hello to Spanish.',
    'What is 15 * 7?',
    'Write a function that adds two numbers in Python.',
    'Is the sky blue? Yes or no.',
]

print('--- Multi-Prompt Verification ---\n')
all_deterministic = True

for i, prompt in enumerate(test_prompts):
    result = engine.verify(prompt=prompt, num_runs=3)
    status = 'PASS' if result.is_deterministic else 'FAIL'
    print(f'  [{status}] Prompt {i+1}: "{prompt[:50]}..."')
    if not result.is_deterministic:
        all_deterministic = False

print(f'\n{"ALL DETERMINISTIC" if all_deterministic else "SOME FAILED"} across {len(test_prompts)} prompts')

## Step 6: Show Environment Fingerprint

In [ ]:
from determl import EnvironmentGuardian

guardian = EnvironmentGuardian()
fingerprint = guardian.create_fingerprint()
print(fingerprint)

## Step 7: Export Proof

This proof can be sent to another node to verify the computation.

In [ ]:
import json

result = engine.run('What is 2 + 2? Answer with just the number.')
proof = result.to_proof()

print('--- Execution Proof ---')
print(json.dumps(proof, indent=2))
print(f'\nAnother node with same model + seed should get canonical_hash: {proof["canonical_hash"]}')